# Short link to this Colab:
# https://bit.ly/fse-tutorial-5

# Information retrieval, bug report reformulation and agentic bug localization
In this tutorial, we look into how reformulating bug reports into a JSON object with only the relevant information improves retrieval.

We'll demonstrate what reformulation looks like with a sample bug report from the SWE-bench Lite dataset.

1) We start by running a BM25 indexer on the code repository we are targetting (done outside of this notebook) and run a BM25 search over the repository using the raw bug report.


2) Then, we look into how reformulating the bug report impacts BM25 retrieval results. We run an agent that performs file-level bug localization using tools: extract relevant information from bug, run BM25, view files. We inspect the generated agent trajectory and reflect on potential issues, error recovery and self-correction.

In [ ]:
from google.colab import userdata
import gdown, zipfile
import os

import json
from IPython.display import JSON
import pandas as pd

# Do not implicitly look for/use HF_TOKEN.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"


In [ ]:
!pip install unidiff
!pip install langchain-text-splitters

!pip install -q --no-deps git+https://github.com/gencau/boatse-extractor.git --force-reinstall

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from boatse_extractor import BugInfoExtractor
from boatse_extractor.prompts.agent_context_prompt import AgentContextPrompt
from utils.json_utils import parse_json_safe

### Baseline: raw issues
Here we take a look at a raw issue example from SWE-bench Lite, a dataset that contains 300 bug resolution tasks.

What could make retrieval challenging with this bug report?

In [ ]:
# Start with a raw problem statement (django-11283)
bug_report = """
Migration auth.0011_update_proxy_permissions fails for models recreated as a proxy.

Description
		(last modified by Mariusz Felisiak)

I am trying to update my project to Django 2.2. When I launch python manage.py migrate,
I get this error message when migration auth.0011_update_proxy_permissions is applying (full stacktrace is available ​here):

django.db.utils.IntegrityError: duplicate key value violates unique constraint "idx_18141_auth_permission_content_type_id_01ab375a_uniq"
DETAIL: Key (co.ntent_type_id, codename)=(12, add_agency) already exists.

It looks like the migration is trying to re-create already existing entries in the auth_permission table.
At first I though it cloud because we recently renamed a model. But after digging and deleting the entries associated with the renamed
model from our database in the auth_permission table, the problem still occurs with other proxy models.
I tried to update directly from 2.0.13 and 2.1.8. The issues appeared each time. I also deleted my venv and recreated it without
an effect.

I searched for a ticket about this on the bug tracker but found nothing.
I also posted this on ​django-users and was asked to report this here.
"""

### Buggy File
Note the buggy file we're looking for:

**`django/contrib/auth/migrations/0011_update_proxy_permissions.py`**

We will refer to it later, and hopefully see it appear in some of our results later!

If found, it will be highlighted in light blue.

### Ask BM25 to locate files relevant to the query.
We make two different search requests: one with the raw bug report, and one with the focused, reformulated bug report.

Note the difference in the top-k list returned.

In [ ]:
# First, setup the environment with Pyserini
!pip install pyserini==0.24.0 faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.1/142.1 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 25.6 MB/s eta 0:00:00


In [ ]:
# Define the repository path
REPO_INDEX_PATH = "/content/django_index/django__django__08a4ee06510ae45562c228eefbdcaac84bd38c7a/indexes"

In [ ]:
# Let's download the sample repo, containing the pre-built index needed by BM25
FILE_ID = "1bXxh6XDKPZRJ9XbDs75oHjmMf_vDB2k1"

gdown.download(id=FILE_ID, output="django_index.zip", quiet=False)

with zipfile.ZipFile("django_index.zip", "r") as z:
    z.extractall("django_index")


Downloading...
From: https://drive.google.com/uc?id=1bXxh6XDKPZRJ9XbDs75oHjmMf_vDB2k1
To: /content/django_index.zip
100%|██████████| 13.5M/13.5M [00:00<00:00, 186MB/s]


In [ ]:
### This is to counter an issue with Pyserini requiring an OpenAI API key to be defined, even if we don't need it.
os.environ['OPENAI_API_KEY'] = "test"

# Import the required search methods from Pyserini
from pyserini.search.lucene import LuceneSearcher
from pyserini.index import IndexReader

In [ ]:
# BM25 search method
def search_index (query, repo_path):
  searcher = LuceneSearcher(repo_path)
  index_reader = IndexReader(repo_path) # Initialize IndexReader

  # Perform search
  # Let's ask for k=10, which returns the 10 best results
  hits = searcher.search(query, k=10)

  # Now read the results and return
  query_results = [{
              'docid': hit.docid,
              'score': hit.score,
              'file_name': (
                  json.loads(index_reader.doc(hit.docid).get('raw')).get('path')
                  if index_reader.doc(hit.docid).get('raw') else None
              )
          } for hit in hits]
  return query_results

In [ ]:
# View the results cleanly
def highlight_ground_truth(row):
    ground_truth_file = 'django/contrib/auth/migrations/0011_update_proxy_permissions.py'
    if row['file_name'] == ground_truth_file:
        return ['background-color: lightblue; color: black'] * len(row)
    return [''] * len(row)

In [ ]:
# Search using the raw bug report first
query_results = search_index(bug_report, REPO_INDEX_PATH)
df_results = pd.DataFrame(query_results)

display(df_results.style.apply(highlight_ground_truth, axis=1))

,docid,score,file_name
0,627,72.479202,django/db/migrations/autodetector.py
1,648,64.895599,django/db/models/base.py
2,680,62.647099,django/db/models/sql/query.py
3,690,62.042801,django/forms/models.py
4,657,60.908100,django/db/models/query.py
5,631,59.714199,django/db/migrations/loader.py
6,665,59.491001,django/db/models/fields/related.py
7,625,58.484200,django/db/backends/sqlite3/schema.py
8,656,57.519199,django/db/models/options.py
9,579,57.453300,django/db/backends/base/creation.py


## Reformulate the Bug Report

We ask an open source model, Qwen3-30B, to reformulate the query, given the schema we define.

In [ ]:
OPEN_ROUTER_API_KEY="YOUR_OPENROUTER_API_KEY"

In [ ]:
# Instantiate the BugInfoExtractor, which will take care of the reformulation.
extractor = BugInfoExtractor(
    prompt=AgentContextPrompt(),
    api_key=OPEN_ROUTER_API_KEY
)

In [ ]:
# Extract the bug report -> this will prompt the model to return a JSON schema with fields:
# - Explanation
# - Path
# - Filename
# - Identifiers
# - Code snippet
# - Stacktrace
# - Error message
result = extractor(bug_report)
parsed = parse_json_safe(result)

# Convert the summarized info into a DataFrame for a cleaner table view
info_dict = parsed.get('summarized_info', {})
df_info = pd.DataFrame(list(info_dict.items()), columns=['Field', 'Value'])

display(df_info)

,Field,Value
0,explanation,The migration auth.0011_update_proxy_permissio...
1,path,migrations/0011_update_proxy_permissions.py
2,filename,0011_update_proxy_permissions.py
3,identifiers,"[auth.0011_update_proxy_permissions, update_pr..."
4,code snippet,
5,stacktrace,
6,error_message,django.db.utils.IntegrityError: duplicate key ...


Notice the path and filename fields filled by the LLM that did the reformulation. This is an indication that the LLM has seen this repository, thus data leakage is at play here. This is a common threat to validity in LLM-based experiments.

### Format the reformulated query
Here, we prepare the query for BM25. We remove all unnecessary tokens and keep it to the essential.

Preliminary tests have shown that the best combination of fields is: explanation, code identifiers and code snippets. Let's use this version here.

You can experiment with adding/removing fields and observe the changes in the returned top-k list.

In [ ]:
# Strip {,}, and field names from summarized_info
def extract_fields(json_obj):
    try:
        info = json_obj.get('summarized_info', '') # Get 'explanation' value, or empty string if not found
        clean_str = (
            info.get("explanation", "") + " " +
            #info.get("path", "") + " " +
            #info.get("filename", "") + " " +
            " ".join(info.get("identifiers", [])) +  " " + # join list to string
            info.get("code snippet", "")
            #info.get("stacktrace", "") + " " +
            #info.get("error_message", "")
        )
        return clean_str
    except json.JSONDecodeError:
        # If for some reason it's not valid JSON, return original string or handle as appropriate
        return json_obj

extracted_fields = extract_fields(parsed)
display(extracted_fields)



"The migration auth.0011_update_proxy_permissions fails due to a duplicate key error in the auth_permission table when trying to insert a permission entry that already exists, specifically for a proxy model's permission. This suggests the migration attempts to recreate permissions that were already added, likely due to incorrect handling of proxy models during migration. auth.0011_update_proxy_permissions update_proxy_permissions auth_permission "

In [ ]:
# Then search again with the reformulated bug report
reformulated_query_results = search_index(extracted_fields, REPO_INDEX_PATH)
df_reformulated_results = pd.DataFrame(reformulated_query_results)

display(df_reformulated_results.style.apply(highlight_ground_truth, axis=1))

,docid,score,file_name
0,223,26.879400,django/contrib/auth/migrations/0011_update_proxy_permissions.py
1,627,26.680300,django/db/migrations/autodetector.py
2,680,25.200899,django/db/models/sql/query.py
3,648,24.750299,django/db/models/base.py
4,656,24.225300,django/db/models/options.py
5,579,23.567400,django/db/backends/base/creation.py
6,210,23.475100,django/contrib/auth/management/__init__.py
7,637,22.973000,django/db/migrations/state.py
8,601,22.098301,django/db/backends/oracle/creation.py
9,190,21.844999,django/contrib/auth/admin.py


### Summary

In the previous example, we can see that using a raw bug report as query, as opposed to a focused, reformulated query, yields a different top-10. This suggests that changes in what's provided as information, even if derived from the same bug report, impacts lexical search.

The ground truth file for this particular task is **django/contrib/auth/migrations/0011_update_proxy_permissions.py**.

We can see that it appears in the top-10 list with the reformulated query, but does not in the raw bug report-based query.

**The retriever did not change. The repository did not change. The only thing that changed was the representation of the bug report**.

--> Results may vary between runs because LLM generation is not perfectly deterministic, even with temperature set to 0.

### Takeaways
- Reformulation moved the ground-truth file from outside the top-10 into the top-5 (results can vary).
- Reformulation can improve lexical retrieval by aligning the query with source-code vocabulary.
- Caveat: this is a single example. We will see the impact over the complete repo in the slides.

# Agent-based localization
In this section, we operationalize the reformulated queries with a small agentic loop.

In our original workflow, we use a smaller, open-source model as it is lightweight enough to be used on a local machine. For this tutorial, we will access the model via an API to reduce load times and ensure a smoother experience.

### Design
Implement a small agentic loop with access to tools:
- Extract relevant information from the bug
- BM25 (k=10, 20, 30)
- View files
- Exit and provide results

The goal is to demonstrate the agent tool selection and usability in a development workflow. Traces show the whole agent trajectory, including tool selections and error mitigation, if applicable.

Demonstration is done on a single issue.

In [ ]:
# Define the repository base path.
# This is used by the self-correction mechanism to validate agent-returned paths.
REPO_BASE_PATH="/content/django_index"

# Define the dataset path
# We will use a single record from this dataset to test the agent
DATASET_ID = "1B30NjyYDKGblhvgKyNUw8sUQxIAelA-T"

gdown.download(id=DATASET_ID, output="data_swe.csv", quiet=False)
DATASET_PATH = "/content/data_swe.csv"

Downloading...
From: https://drive.google.com/uc?id=1B30NjyYDKGblhvgKyNUw8sUQxIAelA-T
To: /content/data_swe.csv
100%|██████████| 3.57M/3.57M [00:00<00:00, 193MB/s]


In [ ]:
from boatse_agent.agent_loop import BoatseAgent

# Now setup the agent
agent = BoatseAgent(
    api_key=OPEN_ROUTER_API_KEY,
    dataset_path=DATASET_PATH,
    repo_base_path=REPO_BASE_PATH,
    issue_index="django__django-11283",
    extracted_info=parsed["summarized_info"],
    extracted_option="best", # Options: best, all from the JSON schema
    top_k=5,
)

In [ ]:
# Now execute the pipeline on the selected issue
agent.localize()

config.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]



=== Processing issue django__django-11283 in django__django__08a4ee06510ae45562c228eefbdcaac84bd38c7a ===
Token count: 566
Token count: 985
BM25 query: The migration auth.0011_update_proxy_permissions fails due to a duplicate key error in the auth_permission table when trying to insert a permission entry that already exists, specifically for a proxy model's permission. This suggests the migration attempts to recreate permissions that were already added, likely due to incorrect handling of proxy models during migration. auth.0011_update_proxy_permissions update_proxy_permissions auth_permission
Retrieving top-10 results with BM25 search...
Top 10 results for 'The migration auth.0011_update_proxy_permissions fails due to a duplicate key error in the auth_permission table when trying to insert a permission entry that already exists, specifically for a proxy model's permission. This suggests the migration attempts to recreate permissions that were already added, likely due to incorrect h

### Viewing results and trajectory

Click on the folder icon in the left pane. You will see two directories:
- Logs: you will find a log of all actions taken by the agent, forming the agent trajectory. We can inspect this file to discover the decisions the agent made, where it went wrong and how it corrected course.
- Results: here you will find a .csv file with the final results (ranked list of files).

#### Results
Let's start with the results.

In [ ]:
import ast
import glob
import os

# Find the latest results file
list_of_files = glob.glob('/content/results/data_swe/**/*_results_data_swe_qwen/*.csv', recursive=True)
latest_file = max(list_of_files, key=os.path.getctime)

print(f"Loading latest results from: {latest_file}")

# Load results
results_df = pd.read_csv(latest_file)

# final_files is stored as a string representation of a list in the CSV, so we parse it
final_files_list = ast.literal_eval(results_df['final_files'].iloc[0])

# Create a DataFrame for structured display
df_final_results = pd.DataFrame(final_files_list, columns=['file_name'])

# Add a numbering column starting at 1
df_final_results.insert(0, 'Rank', range(1, len(df_final_results) + 1))
pd.set_option('display.max_colwidth', None)

display(df_final_results.style.apply(highlight_ground_truth, axis=1).hide(axis='index'))

Loading latest results from: /content/results/data_swe/20260705T151035Z_results_data_swe_qwen/qwen3-30b-a3b-instruct-2507_best_top5.csv


Rank,file_name
1,django/contrib/auth/migrations/0011_update_proxy_permissions.py
2,django/db/migrations/autodetector.py
3,django/db/models/sql/query.py
4,django/db/models/base.py
5,django/db/models/options.py


#### Agent Trajectory
Now, let's take a look at the agent's trajectory.

Inspect the prompts, tool calls and any errors encountered. Were they resolved successfully?

See how the prompt context size grows as the conversation goes on.

In [ ]:
from trajectory_viewer import render_trajectory

# Dynamically find the latest log file
log_files = glob.glob('/content/logs/data_swe/**/*.jsonl', recursive=True)
latest_file = max(log_files, key=os.path.getctime)

render_trajectory(latest_file)



Loaded 49 raw events.


### Some Observations
- The Agentic loop is able to surface the ground truth file in first position.
- In the extract_relevant tool call, the filename was already set to the ground truth, although it is never fully mentioned in the issue description: an indication of data leakage by the model that extracted the information from the bug.

Other observations may vary depending on your run, as it is non-deterministic.

Watch for:
- Erroneous responses from the LLM, marked in red.
- Subsequent corrections by the model (re-prompt), either successful or not.

### Context Management
Context grows with the conversation. In a longer conversation workflow, we would require more advanced context management techniques to avoid the context from growing beyond the context window.

In our case, our LLM uses a 16K **context window** so that it remains manageable on a local machine.

We have used at least **three context management techniques** throughout this Colab:
- Reformulation, to focus on the most relevant information
- Space reduction with BM25, so the LLM doesn't have to inspect the whole repo
- Summarization, so that the LLM can review relevant evidence and revise its answer.

# Summary
- The same BM25 retriever can produce substantially different rankings depending on how the bug report is represented.
- Context construction can matter as much as a better model. A focused query can remove noise from the raw issue and emphasize source-code-relevant terms such as identifiers, error messages, and code snippets.
- BM25 is useful as a context-reduction step. Even a lightweight lexical retriever can narrow a large repository to a smaller set of candidate files for inspection.
- Agentic localization builds on retrieval. The agent does not inspect the whole repository blindly; it uses reformulated bug information, BM25 results, and file-viewing tools to reason over a constrained context.
- Evaluation requires caution. LLM-generated reformulations may introduce nondeterminism or leakage.